In [1]:
%pwd

'd:\\Tipto\\agentic-ai-projects\\hr-policy-agent\\notebooks'

In [2]:
import os 
os.chdir("../")

In [3]:
# markdown file path
from pathlib import Path
markdown_file_path = Path("data/processed/hr_policy.md")
markdown_file_path.exists() 

True

In [4]:
from langchain_community.document_loaders.markdown import UnstructuredMarkdownLoader

loader = UnstructuredMarkdownLoader(markdown_file_path)
documents = loader.load()
len(documents)

C:\Users\tipto\AppData\Local\Temp\ipykernel_17060\1518003062.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.markdown import UnstructuredMarkdownLoader


1

In [5]:
print(documents[0].page_content)

gesci Founded by UN ICT Task Force

Technology • Innovation • Education

HUMAN RESOURCE POLICIES AND PROCEDURE MANUAL (HRPPM)

REVISED 2018

gesci

Founded by UN ICT Task Force

FOREWARD FROM MANAGEMENT

Dear GESCI Team Member,

This manual provides details on GESCI's policies. All members of staff should read and understand them. We urge you to seek any clarification as may be necessary from your Head of Unit, your project Manager and /or the Chief Executive Officer.

The Chief Executive Officer reserves final arbitration on the interpretation of all policies, procedures and practices dealt with in this manual. The Finance unit, handling the Human Resources needs, will act as an advisory unit to ensure that these policies are followed.

The success of GESCI depends on your commitment and professionalism and, combined with team work, will provide the highest level of service to the communities we serve. We look forward to your professional contribution and continuous support for the gr

- Doing Hierarchical/Structural Chunking

In [6]:
# first, remove "gesci Founded by UN ICT Task Force" from the documents
import re
header_pattern = r"(?i)gesci\s*\n?\s*Founded by UN ICT Task Force"


for doc in documents:
    doc.page_content = re.sub(header_pattern, "", doc.page_content).strip()

In [7]:
def txt_to_markdown(text: str) -> str:
    """Convert the GESCI HRPPM text to structured Markdown."""
    lines = text.split("\n")
    md_lines = []
    
    for line in lines:
        stripped = line.strip()
        
        # Skip empty lines but preserve paragraph breaks
        if not stripped:
            md_lines.append("")
            continue
        
        # Detect top-level section headers (e.g., "1. SCOPE AND PURPOSE", "2 DUTIES...")
        if re.match(r'^\d+\.?\s+[A-Z][A-Z\s,&]+$', stripped) and len(stripped) < 80:
            md_lines.append(f"\n# {stripped}\n")
        
        # Detect sub-section headers (e.g., "2.1 Duties, Rights and Obligations of GESCI")
        elif re.match(r'^\d+\.\d+\s+[A-Z]', stripped) and len(stripped) < 100:
            md_lines.append(f"\n## {stripped}\n")
        
        # Detect sub-sub-section headers (e.g., "2.2.1 Prohibited conduct")
        elif re.match(r'^\d+\.\d+\.\d+\s+[A-Z]', stripped) and len(stripped) < 120:
            md_lines.append(f"\n### {stripped}\n")
        
        # Detect sub-sub-sub-section headers (e.g., "3.3.1 Existence of a Vacancy")
        elif re.match(r'^\d+\.\d+\.\d+\.\d+\s+[A-Z]', stripped) and len(stripped) < 120:
            md_lines.append(f"\n#### {stripped}\n")
        
        # Detect ALL CAPS headers without numbers (e.g., "FOREWARD FROM MANAGEMENT")
        elif stripped.isupper() and len(stripped) < 80 and len(stripped.split()) <= 10:
            md_lines.append(f"\n# {stripped}\n")
        
        # Detect title-case headers without numbers (e.g., "Overall Human Resource Policy Statement")
        elif (stripped.istitle() and len(stripped) < 80 
              and not stripped.endswith('.') 
              and len(stripped.split()) <= 8):
            md_lines.append(f"\n## {stripped}\n")
        
        else:
            md_lines.append(stripped)
    
    return "\n".join(md_lines)

In [8]:
# convert the document to markdown format
markdown_content = txt_to_markdown(documents[0].page_content)

In [9]:
print(markdown_content)


## Technology • Innovation • Education



# HUMAN RESOURCE POLICIES AND PROCEDURE MANUAL (HRPPM)



# REVISED 2018





# FOREWARD FROM MANAGEMENT


Dear GESCI Team Member,

This manual provides details on GESCI's policies. All members of staff should read and understand them. We urge you to seek any clarification as may be necessary from your Head of Unit, your project Manager and /or the Chief Executive Officer.

The Chief Executive Officer reserves final arbitration on the interpretation of all policies, procedures and practices dealt with in this manual. The Finance unit, handling the Human Resources needs, will act as an advisory unit to ensure that these policies are followed.

The success of GESCI depends on your commitment and professionalism and, combined with team work, will provide the highest level of service to the communities we serve. We look forward to your professional contribution and continuous support for the growth of GESCI

Signed on behalf of ~~Management~~,

[s

In [10]:
headers_to_split_on = [
    ("#", "Section"),
    ("##", "SubSection"),
    ("###", "SubSubSection"),
    ("####", "SubSubSubSection"),
]

In [11]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on = headers_to_split_on,
    strip_headers = False # keep the headers in the chunks
)
chunks = markdown_splitter.split_text(markdown_content)

In [12]:
for i, chunk in enumerate(chunks):
    print(f"----------------- Chunk {i+1} -----------------")
    print(f"Metadata: {chunk.metadata}")
    print(f"Content:\n{chunk.page_content}\n")
    print()

----------------- Chunk 1 -----------------
Metadata: {'SubSection': 'Technology • Innovation • Education'}
Content:
## Technology • Innovation • Education


----------------- Chunk 2 -----------------
Metadata: {'Section': 'HUMAN RESOURCE POLICIES AND PROCEDURE MANUAL (HRPPM)'}
Content:
# HUMAN RESOURCE POLICIES AND PROCEDURE MANUAL (HRPPM)


----------------- Chunk 3 -----------------
Metadata: {'Section': 'REVISED 2018'}
Content:
# REVISED 2018


----------------- Chunk 4 -----------------
Metadata: {'Section': 'FOREWARD FROM MANAGEMENT'}
Content:
# FOREWARD FROM MANAGEMENT  
Dear GESCI Team Member,  
This manual provides details on GESCI's policies. All members of staff should read and understand them. We urge you to seek any clarification as may be necessary from your Head of Unit, your project Manager and /or the Chief Executive Officer.  
The Chief Executive Officer reserves final arbitration on the interpretation of all policies, procedures and practices dealt with in this manu

In [13]:
# add document level metadata to each chunk
from langchain_core.documents import Document

def enrich_chunks(chunks: list[Document]) -> list[Document]:
    enriched_chunks = []
    for chunk in chunks:
        meta = chunk.metadata
        
        # Build a breadcrumb path from the header hierarchy
        breadcrumb_parts = [
            meta.get("Section", ""),
            meta.get("SubSection", ""),
            meta.get("SubSubSection", ""),
            meta.get("SubSubSubSection", ""),
        ]
        breadcrumb = " > ".join(p for p in breadcrumb_parts if p)
        enriched_meta = {
            **meta,
            "source": "GESCI_HRPPM_2018",
            "document_type": "HR Policy Manual",
            "organization": "GESCI",
            "year": "2018",
            "breadcrumb": breadcrumb,
        }
        
        enriched_chunks.append(Document(
            page_content=chunk.page_content,
            metadata = enriched_meta,
        ))
    return enriched_chunks

In [14]:
enriched_chunks = enrich_chunks(chunks)

In [15]:
# handle large chunks by splitting them into smaller chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 2000,
    chunk_overlap = 200,
    separators = ["\n\n", "\n", ". ", " ", ""],
    length_function = len,
)

In [16]:
def split_large_chunks(chunks: list[Document], max_size: int = 2000) -> list[Document]:
    """Split large chunks into smaller chunks based on the specified max_size."""
    result = []
    for chunk in chunks:
        if len(chunk.page_content) <= max_size:
            result.append(chunk)
        else:
            # Split the chunk into smaller chunks
            sub_chunks = recursive_text_splitter.split_text(chunk.page_content)
            for sub_chunk_index, sub_chunk in enumerate(sub_chunks):
                # Create a new Document for each sub-chunk with the same metadata
                result.append(Document(
                    page_content = sub_chunk,
                    metadata = {
                        **chunk.metadata,
                        "sub_chunk_index": sub_chunk_index,
                    }
                ))
    
    return result

In [17]:
final_chunks = split_large_chunks(enriched_chunks, max_size = 2000)
len(final_chunks)

197

# Doing the Embedding

In [18]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = HuggingFaceEmbeddings(
    model_name = embedding_model_name,
    encode_kwargs = {"normalize_embeddings": True}
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [19]:
sample_embedding = embedding_model.embed_query("This is a sample query for embedding.")
len(sample_embedding)

384

In [20]:
# creating pinecone vector store
from pinecone import Pinecone, ServerlessSpec
import time

In [21]:
import os 
from dotenv import load_dotenv
load_dotenv()

True

In [22]:
INDEX_NAME = "hr-policy-agent"
NAMESPACE = "hr-policy-agent-namespace"
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

# connect to Pinecone
pinecone = Pinecone(
    api_key = PINECONE_API_KEY
)

In [23]:
# create the index if it doesn't exist
existing_indexes = [
    index_info["name"] for index_info in pinecone.list_indexes()
]
existing_indexes

['medical-chatbot', 'hr-policy-agent']

In [24]:
if INDEX_NAME not in existing_indexes:
    # create the index
    pinecone.create_index(
        name = INDEX_NAME,
        dimension = len(sample_embedding),
        metric = "cosine",
        spec = ServerlessSpec(
            cloud = "aws",
            region = "us-east-1",
        )
    )
    
    # wait for the index to be ready
    while not pinecone.describe_index(INDEX_NAME)["ready"]:
        print("Waiting for index to be ready...")
        time.sleep(1)

print(f"Index '{INDEX_NAME}' is ready.")

Index 'hr-policy-agent' is ready.


In [25]:
index = pinecone.Index(INDEX_NAME)

In [26]:
from langchain_pinecone import PineconeVectorStore

vectorstore = PineconeVectorStore.from_documents(
    documents = final_chunks,
    embedding = embedding_model,
    index_name = INDEX_NAME,
    namespace = NAMESPACE,
)

In [27]:
retriever = vectorstore.as_retriever(
    search_kwargs = {
        "k": 5,
        "namespace": NAMESPACE
    }
)

In [28]:
# load existing index
pinecone = Pinecone(
    api_key = PINECONE_API_KEY
)
index = pinecone.Index(INDEX_NAME)
vectorstore = PineconeVectorStore(
    index = index,
    embedding = embedding_model,
    namespace = NAMESPACE
)
retriever = vectorstore.as_retriever(
    search_kwargs = {
        "k": 5,
        "namespace": NAMESPACE
    }
)

In [29]:
sample_query = "What are the duties and obligations of GESCI?"
retrieved_docs = retriever.invoke(
    input = sample_query
)
len(retrieved_docs)

5

In [30]:
for i, doc in enumerate(retrieved_docs):
    print(f"----------------- Retrieved Document {i+1} -----------------")
    print(f"Metadata: {doc.metadata}")
    print(f"Content:\n{doc.page_content}\n")
    print()

----------------- Retrieved Document 1 -----------------
Metadata: {'Section': '2 DUTIES, RIGHTS AND OBLIGATIONS', 'SubSection': '2.2 Duties, Rights and Obligations of Staff', 'breadcrumb': '2 DUTIES, RIGHTS AND OBLIGATIONS > 2.2 Duties, Rights and Obligations of Staff', 'document_type': 'HR Policy Manual', 'organization': 'GESCI', 'source': 'GESCI_HRPPM_2018', 'year': '2018'}
Content:
## 2.2 Duties, Rights and Obligations of Staff  
The nature of GESCI's work and the continued success of GESCI require staff of high quality, integrity, expertise and professionalism. The nature of GESCI's work also requires that staff have a special responsibility to avoid situations and activities that might reflect adversely on GESCI, compromise operations, or lead to real or apparent conflicts of interest. Staff shall therefore:  
a) At all times uphold high standards of the principles of honesty, integrity, hard work, commitment and dedication to work, loyalty to GESCI as employer, justice and fair 